# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshit5445/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

## Lane 2 — Refresh / Content Opportunity Scoring

This baseline ranks content for refresh/review priority. It is decision-support: a high score means the page has measured signals that make it worth inspecting, not that a refresh will definitely recover traffic.

In [13]:
%pip -q install duckdb

import os
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is required. Store it in the Colab Secret named HF_TOKEN.")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN getvariable('hf_token')
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = REL + "/fact_content_daily_performance"
MONTH = "2026-03"
DATA = "read_parquet('" + FACT + "/month=" + MONTH + "/*.parquet')"

print("Warehouse connection ready for", MONTH)

Warehouse connection ready for 2026-03


## 1. My rule and its reason codes

**Rule in plain words:** I rank pages higher when they have substantial measured search visibility and their CTR is low relative to the position they already hold. The idea is to prioritize pages where a visible search result may have room for a content/title/snippet review.

**Signal 1 — CTR versus position:** I use the relationship between average position and CTR to identify pages whose CTR is low for their position bucket.

**Signal 2 — search visibility volume:** I use impressions as an impact signal so that a small CTR opportunity on a page seen often is easier to prioritize than the same-sized opportunity on a rarely seen page.

**Reason code:** `LOW_CTR_FOR_POSITION`

The rule has one reason code only. The action label is `REFRESH_REVIEW` for pages that meet the rule; other measured pages remain `MONITOR`.

**Decision-time boundary:** every score input comes from March 2026 only. No April/future outcome or label-derived field is used.

In [14]:
# Signal 1: CTR versus average position.
# The bucket table is the evidence used for the first signal.
signal_position = con.sql(f"""
WITH page AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS impressions,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS clicks,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_sum_position ELSE 0 END)
            / NULLIF(SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END), 0) AS avg_position
    FROM {DATA}
    GROUP BY 1, 2
), scored AS (
    SELECT *,
           CASE
               WHEN avg_position <= 3 THEN 'top_3'
               WHEN avg_position <= 10 THEN 'page_1'
               WHEN avg_position <= 20 THEN 'page_2'
               WHEN avg_position <= 50 THEN 'page_3_5'
               ELSE 'deep'
           END AS position_bucket,
           CASE WHEN impressions > 0 THEN clicks / impressions ELSE 0 END AS ctr
    FROM page
    WHERE impressions > 0
)
SELECT
    position_bucket,
    COUNT(*) AS n,
    ROUND(AVG(ctr), 5) AS mean_ctr,
    ROUND(MEDIAN(ctr), 5) AS median_ctr
FROM scored
GROUP BY 1
ORDER BY CASE position_bucket
    WHEN 'top_3' THEN 1 WHEN 'page_1' THEN 2 WHEN 'page_2' THEN 3
    WHEN 'page_3_5' THEN 4 ELSE 5 END
""").df()

display(signal_position)

# A data-driven verdict: CONFIRMED when mean and median CTR decline across
# progressively worse position buckets, allowing small ties/noise.
position_order = ['top_3', 'page_1', 'page_2', 'page_3_5', 'deep']
position_observed = signal_position.set_index('position_bucket').reindex(position_order).dropna()
mean_non_increasing = position_observed['mean_ctr'].diff().dropna().le(0).mean() >= 0.75
median_non_increasing = position_observed['median_ctr'].diff().dropna().le(0).mean() >= 0.75
signal_1_verdict = 'CONFIRMED' if mean_non_increasing and median_non_increasing else ('MIXED' if mean_non_increasing or median_non_increasing else 'OPPOSITE')
print('Signal 1 verdict:', signal_1_verdict)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,mean_ctr,median_ctr
0,top_3,18860,0.01170,0.0
1,page_1,83288,0.00487,0.0
2,page_2,29922,0.00328,0.0
3,page_3_5,32240,0.00238,0.0
4,deep,12428,0.00085,0.0


Signal 1 verdict: CONFIRMED


In [15]:
# Signal 2: search visibility volume.
# Higher impression buckets are checked against observed click volume.
signal_volume = con.sql(f"""
WITH page AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS impressions,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END) AS clicks
    FROM {DATA}
    GROUP BY 1, 2
), bucketed AS (
    SELECT *,
           CASE
               WHEN impressions = 0 THEN 'zero'
               WHEN impressions <= 100 THEN '1-100'
               WHEN impressions <= 1000 THEN '101-1k'
               WHEN impressions <= 10000 THEN '1k-10k'
               ELSE '10k+'
           END AS impression_bucket
    FROM page
)
SELECT
    impression_bucket,
    COUNT(*) AS n,
    ROUND(AVG(impressions), 1) AS mean_impressions,
    ROUND(AVG(clicks), 2) AS mean_clicks
FROM bucketed
GROUP BY 1
ORDER BY CASE impression_bucket
    WHEN 'zero' THEN 1 WHEN '1-100' THEN 2 WHEN '101-1k' THEN 3
    WHEN '1k-10k' THEN 4 ELSE 5 END
""").df()

display(signal_volume)

volume_observed = signal_volume[signal_volume['impression_bucket'] != 'zero'].copy()
volume_observed['bucket_mid_order'] = volume_observed['impression_bucket'].map({'1-100':1,'101-1k':2,'1k-10k':3,'10k+':4})
volume_observed = volume_observed.sort_values('bucket_mid_order')
clicks_increase = volume_observed['mean_clicks'].diff().dropna().ge(0).mean() >= 0.67
signal_2_verdict = 'CONFIRMED' if clicks_increase else ('MIXED' if volume_observed['mean_clicks'].corr(volume_observed['bucket_mid_order']) > 0 else 'OPPOSITE')
print('Signal 2 verdict:', signal_2_verdict)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_bucket,n,mean_impressions,mean_clicks
0,zero,154699,0.0,0.00
1,1-100,75506,24.9,0.09
2,101-1k,56197,390.9,0.94
3,1k-10k,39158,3262.7,9.82
4,10k+,5877,21957.4,64.29


Signal 2 verdict: CONFIRMED


### Signal verdicts

The code above prints the one-word verdict for each signal from the measured March 2026 bucket tables. I treat these as directional evidence, not as proof that a refresh will improve traffic.

## 2. Build the ranked queue (writes the CSV)

The score combines two verified ideas: relative CTR opportunity for the page's position and search visibility volume. The rule is intentionally simple so the Week-5 model has a transparent baseline to beat.

In [16]:
# Build the ranked baseline queue

# Remove previously-created scoring columns so the cell is safe to re-run.
derived_columns = [
    'benchmark_ctr',
    'ctr_gap',
    'ctr_opportunity_score',
    'volume_score',
    'score',
    'reason_code',
    'action_label'
]

base = base.drop(
    columns=[c for c in derived_columns if c in base.columns]
)

# Use the observed MEAN CTR for each position bucket as the benchmark.
bench = (
    base.groupby(
        'position_bucket',
        observed=False
    )['ctr']
    .mean()
    .rename('benchmark_ctr')
)

# Add the position-specific benchmark to every row.
base = base.join(
    bench,
    on='position_bucket'
)

# Positive gap = CTR is below the observed benchmark
# for pages in the same position bucket.
base['ctr_gap'] = (
    base['benchmark_ctr'] - base['ctr']
).clip(lower=0)

# Normalize the CTR opportunity.
max_gap = base['ctr_gap'].max()

if max_gap > 0:
    base['ctr_opportunity_score'] = (
        base['ctr_gap'] / max_gap
    )
else:
    base['ctr_opportunity_score'] = 0.0

# Normalize impression volume.
max_impressions = base['impressions'].max()

if max_impressions > 0:
    base['volume_score'] = (
        base['impressions'] / max_impressions
    )
else:
    base['volume_score'] = 0.0

# Baseline score:
# 70% CTR opportunity + 30% visibility/volume.
base['score'] = (
    100
    * (
        0.70 * base['ctr_opportunity_score']
        + 0.30 * base['volume_score']
    )
)

# One reason code.
base['reason_code'] = np.where(
    base['ctr_gap'] > 0,
    'LOW_CTR_FOR_POSITION',
    'NO_STRONG_SIGNAL'
)

# Action label.
base['action_label'] = np.where(
    base['ctr_gap'] > 0,
    'REFRESH_REVIEW',
    'MONITOR'
)

# Build the ranked queue.
queue = (
    base[
        [
            'client_hash_id',
            'content_hash_id',
            'score',
            'reason_code',
            'action_label',
            'impressions',
            'ctr',
            'avg_position'
        ]
    ]
    .sort_values(
        ['score', 'impressions'],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue.insert(
    0,
    'rank',
    np.arange(1, len(queue) + 1)
)

# Write the required CSV.
output_path = 'work/outputs/baseline_action_score.csv'

queue.to_csv(
    output_path,
    index=False
)

print(f'Rows ranked: {len(queue)}')
print(f'CSV written: {output_path}')

display(queue.head(10))

Rows ranked: 176738
CSV written: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,reason_code,action_label,impressions,ctr,avg_position
0,1,client_23a62021009f63c4,content_44f34c0a90047651,79.649253,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,212404.0,0.000113,0.665877
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,76.517584,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,134984.0,0.000007,2.693038
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,75.983371,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,124075.0,0.000008,0.308426
3,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,74.003998,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,83834.0,0.000012,0.116003
4,5,client_23a62021009f63c4,content_bf078007df823490,72.173323,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,44707.0,0.000000,1.400049
5,6,client_1a730cb2640a1abf,content_d61fc394d10cba41,71.689779,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,38000.0,0.000026,2.362579
6,7,client_e547b89c05043229,content_8d7d99f109e19aa2,71.392831,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,203497.0,0.001420,2.468557
7,8,client_e547b89c05043229,content_306bc78dff1eb683,71.337086,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,80821.0,0.000433,1.444266
8,9,client_62f4a7e64f5e0096,content_fc67675904376267,71.134754,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,60172.0,0.000299,2.126022
9,10,client_e547b89c05043229,content_dc91779c3d085398,71.012137,LOW_CTR_FOR_POSITION,REFRESH_REVIEW,25625.0,0.000039,2.389151


## 3. Top-20 review

The table below is generated from the ranked queue. The confidence note is deliberately cautious because the score is a hand-written decision rule, not a causal estimate. The final column records a concrete reason the pick could be wrong.

In [17]:
top20 = queue.head(20).copy()

def confidence_note(row):
    if row['action_label'] == 'REFRESH_REVIEW' and row['impressions'] >= base['impressions'].median():
        return 'Higher measured visibility and a position-relative CTR gap support review.'
    if row['action_label'] == 'REFRESH_REVIEW':
        return 'CTR gap is present, but measured visibility is below the page-level median.'
    return 'The rule does not find a strong refresh signal.'

def wrong_if(row):
    if row['avg_position'] > 20:
        return 'It could be a low-value deep result where CTR is mainly explained by position.'
    if row['impressions'] < base['impressions'].median():
        return 'The observed opportunity may be too small to justify refresh effort.'
    return 'The CTR gap may reflect query mix, intent, SERP features, or measurement noise rather than a content problem.'

review = top20.apply(
    lambda r: pd.Series({
        'rank': int(r['rank']),
        'action': r['action_label'],
        'reason_code': r['reason_code'],
        'confidence_note': confidence_note(r),
        'what_would_make_it_wrong': wrong_if(r)
    }), axis=1
)

display(review)

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."
1,2,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."
2,3,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."
3,4,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."
4,5,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."
5,6,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."
6,7,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."
7,8,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."
8,9,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."
9,10,REFRESH_REVIEW,LOW_CTR_FOR_POSITION,Higher measured visibility and a position-rela...,"The CTR gap may reflect query mix, intent, SER..."


## 4. Weak picks + leakage check

A weak pick is one where the rule can be technically satisfied but the decision case is less convincing. I will flag top-ranked rows with relatively low impressions and any rows whose position is outside the main search-result range. I also check that the feature columns do not contain future outcome or label-derived fields.

In [18]:
# Weak-pick review: these are candidates to question, not automatic failures.
weak = queue.head(20).copy()
weak['weak_reason'] = np.select(
    [
        weak['impressions'] < base['impressions'].median(),
        weak['avg_position'] > 20
    ],
    [
        'Below-median visibility makes the action impact less certain.',
        'Low CTR may be largely explained by a deep average position.'
    ],
    default='No single weak-pick flag from the simple checks.'
)
print('Top-20 weak-pick checks:')
display(weak[['rank','score','action_label','impressions','avg_position','weak_reason']])

# Leakage check: only March feature-window fields used by the baseline may appear here.
feature_columns = ['impressions', 'clicks', 'avg_position', 'ctr', 'position_bucket', 'benchmark_ctr', 'ctr_gap']
forbidden_terms = [
    'went_dark',
    'clicks_mar',
    'impressions_mar',
    'sessions_mar',
    'future',
    'outcome'
]
forbidden_present = [c for c in queue.columns if any(term in c.lower() for term in forbidden_terms)]

assert not forbidden_present, f'Forbidden/future fields found: {forbidden_present}'
assert 'went_dark' not in base.columns
assert 'clicks_mar' not in base.columns
assert 'ga4_sessions_mar' not in base.columns

print('Forbidden future/label-derived columns:', forbidden_present)
print('Leakage check passed.')

Top-20 weak-pick checks:


,rank,score,action_label,impressions,avg_position,weak_reason
0,1,79.649253,REFRESH_REVIEW,212404.0,0.665877,No single weak-pick flag from the simple checks.
1,2,76.517584,REFRESH_REVIEW,134984.0,2.693038,No single weak-pick flag from the simple checks.
2,3,75.983371,REFRESH_REVIEW,124075.0,0.308426,No single weak-pick flag from the simple checks.
3,4,74.003998,REFRESH_REVIEW,83834.0,0.116003,No single weak-pick flag from the simple checks.
4,5,72.173323,REFRESH_REVIEW,44707.0,1.400049,No single weak-pick flag from the simple checks.
5,6,71.689779,REFRESH_REVIEW,38000.0,2.362579,No single weak-pick flag from the simple checks.
6,7,71.392831,REFRESH_REVIEW,203497.0,2.468557,No single weak-pick flag from the simple checks.
7,8,71.337086,REFRESH_REVIEW,80821.0,1.444266,No single weak-pick flag from the simple checks.
8,9,71.134754,REFRESH_REVIEW,60172.0,2.126022,No single weak-pick flag from the simple checks.
9,10,71.012137,REFRESH_REVIEW,25625.0,2.389151,No single weak-pick flag from the simple checks.


Forbidden future/label-derived columns: []
Leakage check passed.


## Self-check

- Every section is filled with Markdown reasoning and executable code.
- The notebook must run top to bottom with no errors using **Runtime → Run all**.
- No client names, URLs, or private queries are included.
- Claims use careful words such as observed, measured, directional, and decision-support.
- The required output is regenerated at `work/outputs/baseline_action_score.csv`; the CSV should stay out of git.
- Commit this notebook under `work/notebooks/w04_baseline_score.ipynb` and submit the repo URL.